In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import sys
import logging
import os
from matplotlib.patches import Circle
# sys.path.append(os.path.join('..','..','..'))
path_to_angelFISH = r'C:\Users\formanj\GitHub'
sys.path.append(path_to_angelFISH)


from AngelFISH.src import Receipt, load_data, silence
from AngelFISH.src.Steps import (download_data, segment, detect_spots, get_cell_properties, 
                       return_data, match_masks, export_images, filter_csv, 
                       calculate_sharpness, reconcile_data)
silence()


### Load Data

In [ ]:
receipt = Receipt(
    analysis_name = '09042025_180minTunedPipeline',
    nas_location = None,
    local_location = os.getcwd(), # the idea is that you in an already processed dir
    data_loader = 'recursive_pycromanager_data_loader' # pycromanager_data_loader, recursive_pycromanager_data_loader
    )

receipt.arguments['recursive_analysis_name'] = '09082025_IntronDiffusionPipeline_Combination_littleSticter' # this is the analysis that will be loaded in all of the sub datasets

In [ ]:
data = load_data(receipt)

In [ ]:
for key in data.keys():
    print(key)

In [ ]:
cells = data['introns_cellCounts']
cells

In [ ]:
clusters = data['introns_clusters']
clusters

In [ ]:
ts = data['introns_ts']
ts

## Visualize unfiltered Data

In [ ]:
mean_rna = cells['nb_rna'].mean()
std_rna = cells['nb_rna'].std()
median_rna = cells['nb_rna'].median()

plt.hist(cells['nb_rna'], bins=50, alpha=0.7)
plt.axvline(mean_rna, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {mean_rna:.2f}')
plt.axvline(median_rna, color='b', linestyle='dashed', linewidth=2, label=f'Median: {median_rna:.2f}')
plt.axvline(mean_rna + std_rna, color='g', linestyle='dotted', linewidth=2, label=f'+1 STD: {mean_rna + std_rna:.2f}')
plt.axvline(mean_rna - std_rna, color='g', linestyle='dotted', linewidth=2, label=f'-1 STD: {mean_rna - std_rna:.2f}')
plt.xlabel('Number of Spots')
plt.ylabel('Frequency')
plt.title('Number of RNA per cell')
plt.legend()
plt.show()

In [ ]:
mean_spots = clusters['nb_rna'].mean()
std_spots = clusters['nb_rna'].std()
median_spots = clusters['nb_rna'].median()

plt.hist(clusters['nb_rna'], alpha=0.7) # bins=np.arange(0, 81, 1),
plt.axvline(mean_spots, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {mean_spots:.2f}')
plt.axvline(median_spots, color='b', linestyle='dashed', linewidth=2, label=f'Median: {median_spots:.2f}')
plt.axvline(mean_spots + std_spots, color='g', linestyle='dotted', linewidth=2, label=f'+1 STD: {mean_spots + std_spots:.2f}')
plt.axvline(mean_spots - std_spots, color='g', linestyle='dotted', linewidth=2, label=f'-1 STD: {mean_spots - std_spots:.2f}')
plt.xlabel('Number of Spots')
plt.ylabel('Frequency')
plt.title('RNA per nuclear cluster')
plt.legend()
plt.show()

In [ ]:
mean_nuc_area = cells['nuc_area'].mean()
std_nuc_area = cells['nuc_area'].std()
median_nuc_area = cells['nuc_area'].median()

plt.hist(cells['nuc_area'], bins=50, alpha=0.7)
plt.axvline(mean_nuc_area, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {mean_nuc_area:.2f}')
plt.axvline(median_nuc_area, color='b', linestyle='dashed', linewidth=2, label=f'Median: {median_nuc_area:.2f}')
plt.axvline(mean_nuc_area + std_nuc_area, color='g', linestyle='dotted', linewidth=2, label=f'+1 STD: {mean_nuc_area + std_nuc_area:.2f}')
plt.axvline(mean_nuc_area - std_nuc_area, color='g', linestyle='dotted', linewidth=2, label=f'-1 STD: {mean_nuc_area - std_nuc_area:.2f}')
plt.xlabel('Nuclear Size (px)')
plt.ylabel('Frequency')
plt.title('Nucleus size')
plt.legend()
plt.show()

### Filter Data

In [ ]:
# filter cells that are too small or too large
lower_th = mean_nuc_area - 2*std_nuc_area
upper_th = mean_nuc_area + 2*std_nuc_area
cells = cells[(cells['nuc_area'] > lower_th) & (cells['nuc_area'] < upper_th)]

In [ ]:
# filter cells with extreme numbers of rna 
# rna_upper = 150
# cells = cells[cells['nb_rna'] < rna_upper]

In [ ]:
# filter cells with extreme numbers of rna 
# rna_upper = 400
# ts = ts[ts['nb_rna'] < rna_upper]

In [ ]:
# from scipy.spatial.distance import cdist
# import networkx as nx
# import pandas as pd

# # merge neighboring cluster with weighted distances
# distance_threshold = 10

# def merge_clusters(ts, distance_threshold):
#     merged_rows = []
#     # weights for pixel size: x/y = 130 nm, z = 500 nm
#     xy_nm = 130
#     z_nm = 500
#     # scale factors to equalize distances
#     z_weight = z_nm / xy_nm 
#     # Group by fov and experimental_time
#     for (fov, exp_time), group in ts.groupby(['fov', 'experiment_name']):
#         coords = group[['z_px', 'y_px', 'x_px']].values.astype(float)
#         # scale z dimension
#         coords[:, 0] *= z_weight
#         indices = group.index.values
#         # Compute adjacency matrix
#         dist_matrix = cdist(coords, coords)
#         # Build graph
#         G = nx.Graph()
#         for i, idx_i in enumerate(indices):
#             G.add_node(idx_i)
#             for j, idx_j in enumerate(indices):
#                 if i != j and dist_matrix[i, j] <= distance_threshold:
#                     G.add_edge(idx_i, idx_j)
#         # Find connected components (clusters to merge)
#         for component in nx.connected_components(G):
#             rows = group.loc[list(component)]
#             merged = rows.iloc[0].copy()
#             merged['nb_spots'] = rows['nb_spots'].sum()
#             merged['cluster_index'] = list(rows['cluster_index'])
#             merged_rows.append(merged)
#     return pd.DataFrame(merged_rows)

# ts = merge_clusters(clusters, distance_threshold)
# ts

In [ ]:
mean_rna = cells['nb_rna'].mean()
std_rna = cells['nb_rna'].std()
median_rna = cells['nb_rna'].median()

plt.hist(cells['nb_rna'], bins=50, alpha=0.7)
plt.axvline(mean_rna, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {mean_rna:.2f}')
plt.axvline(median_rna, color='b', linestyle='dashed', linewidth=2, label=f'Median: {median_rna:.2f}')
plt.axvline(mean_rna + std_rna, color='g', linestyle='dotted', linewidth=2, label=f'+1 STD: {mean_rna + std_rna:.2f}')
plt.axvline(mean_rna - std_rna, color='g', linestyle='dotted', linewidth=2, label=f'-1 STD: {mean_rna - std_rna:.2f}')
plt.xlabel('Number of Spots')
plt.ylabel('Frequency')
plt.title('Number of RNA per cell')
plt.legend()
plt.show()

In [ ]:
mean_spots = ts['nb_rna'].mean()
std_spots = ts['nb_rna'].std()
median_spots = ts['nb_rna'].median()

plt.hist(ts['nb_rna'], alpha=0.7) # bins=np.arange(0, 81, 1),
plt.axvline(mean_spots, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {mean_spots:.2f}')
plt.axvline(median_spots, color='b', linestyle='dashed', linewidth=2, label=f'Median: {median_spots:.2f}')
plt.axvline(mean_spots + std_spots, color='g', linestyle='dotted', linewidth=2, label=f'+1 STD: {mean_spots + std_spots:.2f}')
plt.axvline(mean_spots - std_spots, color='g', linestyle='dotted', linewidth=2, label=f'-1 STD: {mean_spots - std_spots:.2f}')
plt.xlabel('Number of Spots')
plt.ylabel('Frequency')
plt.title('RNA per nuclear cluster')
plt.legend()
plt.show()

In [ ]:
mean_nuc_area = cells['nuc_area'].mean()
std_nuc_area = cells['nuc_area'].std()
median_nuc_area = cells['nuc_area'].median()

plt.hist(cells['nuc_area'], bins=50, alpha=0.7)
plt.axvline(mean_nuc_area, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {mean_nuc_area:.2f}')
plt.axvline(median_nuc_area, color='b', linestyle='dashed', linewidth=2, label=f'Median: {median_nuc_area:.2f}')
plt.axvline(mean_nuc_area + std_nuc_area, color='g', linestyle='dotted', linewidth=2, label=f'+1 STD: {mean_nuc_area + std_nuc_area:.2f}')
plt.axvline(mean_nuc_area - std_nuc_area, color='g', linestyle='dotted', linewidth=2, label=f'-1 STD: {mean_nuc_area - std_nuc_area:.2f}')
plt.xlabel('Nuclear Size (px)')
plt.ylabel('Frequency')
plt.title('Nucleus size')
plt.legend()
plt.show()

### Display Timeseries

In [ ]:
grouped = cells.groupby('experimental_time')['nb_rna']
mean_nb_rna = grouped.mean()
std_nb_rna = grouped.std()

x = mean_nb_rna.index.values
y = mean_nb_rna.values

def exp_decay(t, A, tau):
    return A * np.exp(-t / tau) # + C

p0 = [y[0], 60]

params, cov = curve_fit(exp_decay, x, y, p0=p0)
A, tau = params

x_fit = np.linspace(x.min(), x.max(), 100)
y_fit = exp_decay(x_fit, A, tau)

plt.errorbar(x, y, yerr=std_nb_rna.values, fmt='o-', capsize=5, label='Data')
plt.plot(x_fit, y_fit, 'r--', label=f'Exp fit: A={A:.1f}, tau={tau:.1f}')

plt.xlabel('Experimental Time (min)')
plt.ylabel('Number of RNA per Cell')
plt.title('nb_rna vs Experimental Time')
plt.legend()
plt.show()

In [ ]:
ts_counts = ts.groupby('experimental_time').size()
cell_counts = cells.groupby('experimental_time').size()
ts_normalized = ts_counts / cell_counts

plt.plot(ts_normalized.index, ts_normalized.values)
plt.xlabel('Experimental Time (min)')
plt.ylabel('Number of Cluster per cell')
plt.title('Number of TS per cell vs Experimental Time')
plt.show()

In [ ]:
RNAperTS_counts = ts.groupby('experimental_time')['nb_rna']

mean_RNAperTS = RNAperTS_counts.mean()
std_RNAperTS = RNAperTS_counts.std()
median_RNAperTS = RNAperTS_counts.median()

plt.errorbar(mean_RNAperTS.index.values, mean_RNAperTS.values, yerr=std_RNAperTS.values, fmt='o-', capsize=5, label='Data')

plt.xlabel('Experimental Time (min)')
plt.ylabel('Number of RNA per TS')
plt.title('Number of RNA per TS vs Experimental Time')
# plt.legend()
plt.show()

In [ ]:
np.log10(1/tau)

In [ ]:
unique_times = cell_counts.index
n_times = len(unique_times)
fig, axes = plt.subplots(n_times, 1, figsize=(8, 3 * n_times), sharex=True)
for i, t in enumerate(unique_times):
    subset = cells[cells['experimental_time'] == t]['nb_rna']
    axes[i].hist(subset, bins=100, alpha=0.7, density=True)
    n_entries = len(subset)
    axes[i].set_title(f'RNA per cell at Time {t} min (n={n_entries})')
    axes[i].set_ylabel('Frequency')
    axes[i].set_ylim(0, 0.1)  # set the same y-axis limits for all subplots
    axes[i].set_xlim(0, 300)

axes[-1].set_xlabel('Number of RNA per cell')
plt.tight_layout()
plt.show()

### Display example cells

In [ ]:
rng = 43
import random 
random.seed(rng)
np.random.seed(rng)
map_Row2FOV = lambda row: data['map_np2p'](row['concat_index'].iloc[0], row['fov'].iloc[0])
cell_props = data['cellproperties']
spots = data['introns_spots']


In [ ]:
random_cell = cells.sample(n=1)

match = cell_props[
    (cell_props['fov'] == random_cell['fov'].iloc[0]) &
    (cell_props['concat_index'] == random_cell['concat_index'].iloc[0]) &
    (cell_props['nuc_area'] == random_cell['nuc_area'].iloc[0])
]

bb = [match['nuc_bbox-0'].iloc[0], match['nuc_bbox-1'].iloc[0], match['nuc_bbox-2'].iloc[0], match['nuc_bbox-3'].iloc[0]]

fovspots = spots[
    (spots['fov'] == random_cell['fov'].iloc[0]) &
    (spots['concat_index'] == random_cell['concat_index'].iloc[0]) &
    (spots['nuc label'] == random_cell['cell_id'].iloc[0])
]
fovTS = ts[
    (ts['fov'] == random_cell['fov'].iloc[0]) &
    (ts['concat_index'] == random_cell['concat_index'].iloc[0])
]
zyx_spots = fovspots[['z (px)', 'y (px)', 'x (px)']].values
zyx_TS = fovTS[['z (px)', 'y (px)', 'x (px)']].values

# assert len(fovTS) == random_cell['nb_transcription_site'].iloc[0]
assert len(fovspots) == random_cell['nb_rna'].iloc[0]

fov = map_Row2FOV(random_cell)

img = data['images'][fov, 0].compute()
mask = data['nuc_masks'][fov, 0].compute()
channel = 0

fig, ax = plt.subplots()
img_slice = np.max(img[channel], axis=0)[bb[0]:bb[2], bb[1]:bb[3]]
vmin, vmax = np.percentile(img_slice, [5, 95])
ax.imshow(img_slice, cmap='gray', vmin=vmin, vmax=vmax)


# Overlay the nucleus mask outline
mask_slice = np.max(mask, axis=0)[bb[0]:bb[2], bb[1]:bb[3]]
contours = plt.contour(mask_slice, levels=[0.5], colors='cyan', linewidths=1, linestyles='solid')

# Draw circles for spots within the bounding box
for z, y, x in zyx_spots:
    # Adjust coordinates to bbox-local
    y_local = y - bb[0]
    x_local = x - bb[1]
    # Only draw if inside the cropped image
    if 0 <= y_local < img_slice.shape[0] and 0 <= x_local < img_slice.shape[1]:
        circ = Circle((x_local, y_local), radius=4, edgecolor='red', facecolor='none', linewidth=1)
        ax.add_patch(circ)

for z, y, x in zyx_TS:
    # Adjust coordinates to bbox-local
    y_local = y - bb[0]
    x_local = x - bb[1]
    # Only draw if inside the cropped image
    if 0 <= y_local < img_slice.shape[0] and 0 <= x_local < img_slice.shape[1]:
        circ = Circle((x_local, y_local), radius=4, edgecolor='blue', facecolor='none', linewidth=1)
        ax.add_patch(circ)

plt.show()
print(f'fov: {random_cell['fov'].iloc[0]}')
print(f'num rna: {random_cell['nb_rna'].iloc[0]}')
print(f'num ts: {random_cell['nb_transcription_site'].iloc[0]}')

In [ ]:
cell_w_ts = cells[cells['nb_transcription_site'] >= 1]

In [ ]:
random_cell = cell_w_ts.sample(n=1)

match = cell_props[
    (cell_props['fov'] == random_cell['fov'].iloc[0]) &
    (cell_props['concat_index'] == random_cell['concat_index'].iloc[0]) &
    (cell_props['nuc_area'] == random_cell['nuc_area'].iloc[0])
]

bb = [match['nuc_bbox-0'].iloc[0], match['nuc_bbox-1'].iloc[0], match['nuc_bbox-2'].iloc[0], match['nuc_bbox-3'].iloc[0]]

fovspots = spots[
    (spots['fov'] == random_cell['fov'].iloc[0]) &
    (spots['concat_index'] == random_cell['concat_index'].iloc[0]) &
    (spots['nuc label'] == random_cell['cell_id'].iloc[0])
]
fovTS = ts[
    (ts['fov'] == random_cell['fov'].iloc[0]) &
    (ts['concat_index'] == random_cell['concat_index'].iloc[0])
]
zyx_spots = fovspots[['z (px)', 'y (px)', 'x (px)']].values
zyx_TS = fovTS[['z (px)', 'y (px)', 'x (px)']].values

# assert len(fovTS) == random_cell['nb_transcription_site'].iloc[0]
# assert len(fovspots) == random_cell['nb_rna'].iloc[0]

fov = map_Row2FOV(random_cell)

img = data['images'][fov, 0].compute()
mask = data['nuc_masks'][fov, 0].compute()
channel = 0

fig, ax = plt.subplots()
img_slice = np.max(img[channel], axis=0)[bb[0]:bb[2], bb[1]:bb[3]]
vmin, vmax = np.percentile(img_slice, [5, 95])
ax.imshow(img_slice, cmap='gray', vmin=vmin, vmax=vmax)

# Overlay the nucleus mask outline
mask_slice = np.max(mask, axis=0)[bb[0]:bb[2], bb[1]:bb[3]]
contours = plt.contour(mask_slice, levels=[0.5], colors='cyan', linewidths=1, linestyles='solid')

# Draw circles for spots within the bounding box
for z, y, x in zyx_spots:
    # Adjust coordinates to bbox-local
    y_local = y - bb[0]
    x_local = x - bb[1]
    # Only draw if inside the cropped image
    if 0 <= y_local < img_slice.shape[0] and 0 <= x_local < img_slice.shape[1]:
        circ = Circle((x_local, y_local), radius=4, edgecolor='red', facecolor='none', linewidth=1)
        ax.add_patch(circ)

for z, y, x in zyx_TS:
    # Adjust coordinates to bbox-local
    y_local = y - bb[0]
    x_local = x - bb[1]
    # Only draw if inside the cropped image
    if 0 <= y_local < img_slice.shape[0] and 0 <= x_local < img_slice.shape[1]:
        circ = Circle((x_local, y_local), radius=4, edgecolor='blue', facecolor='none', linewidth=1)
        ax.add_patch(circ)

plt.show()
print(f'fov: {random_cell['fov'].iloc[0]}')
print(f'num rna: {random_cell['nb_rna'].iloc[0]}')
print(f'num ts: {random_cell['nb_transcription_site'].iloc[0]}')